# ATLAS tutorial: download solar radiation data from Copernicus CDS

This notebook downloads hourly reanalysis data from the Copernicus Climate Data Store (CDS), one month at a time, and saves each month as a separate NetCDF file.

It is designed as a step-by-step tutorial for users who are not Python experts. In most cases, users only need to edit the **Input parameters** section and then run the notebook from top to bottom.

## What this notebook does

1. Installs/imports the required Python packages.
2. Defines the input parameters: product, variable, country/area, dates and output folder.
3. Defines helper functions used for the download.
4. Runs the download.

## Before running

You need a Copernicus CDS account and a CDS API key. Do **not** share your personal API key publicly.

Recommended option: configure your CDS key in the standard CDS configuration file on your machine (`~/.cdsapirc`). If that is already configured, you can leave `CDS_API_KEY = None` below.

Alternative option: paste your key in the `CDS_API_KEY` parameter cell below, but only in your private local copy of this notebook.

## Step 1. Import required packages

Run this cell first. If `cdsapi` is not installed, install it with:

```bash
pip install cdsapi
```

In [1]:
from pathlib import Path
from datetime import datetime, timedelta

import cdsapi

## Step 2. Input parameters

Edit only this section for normal use.

### CDS API access

- `CDS_API_KEY`: your personal CDS API key. Leave it as `None` if your key is already configured in `~/.cdsapirc`.
- `CDS_API_URL`: CDS API endpoint. Usually this should not be changed.

### Dataset selection

- `PRODUCT`: choose `"era5"` or `"era5land"`.
- `VARIABLE`: CDS variable name to download. The default is solar radiation used by the ATLAS workflow.
#### Documentation:

- `"era5"`: check "Variable name on CDS" https://confluence.ecmwf.int/display/CKB/ERA5%3A+data+documentation
- `"era5land"`: check "Varaible name on CDS" https://confluence.ecmwf.int/display/CKB/ERA5-Land%3A+data+documentation

### Country and area

- `COUNTRY`: used to select a predefined bounding box.
- `AREAS`: dictionary of country bounding boxes in CDS format: `[North, West, South, East]`.

To add a new country, add a new entry to `AREAS` and set `COUNTRY` to the same name.

### Dates

- `START_DATE`: first date to download.
- `END_DATE`: last date to download.

The data are downloaded month by month. Each output file contains one month.

### Output folder

- `OUTPUT_DIR`: folder where NetCDF files will be saved. By default, files are saved in a local `data/cds_downloads/...` folder next to this notebook. This avoids using server-specific paths.

In [8]:
# =========================
# CDS API ACCESS
# =========================

# Leave as None if your CDS API key is already configured in ~/.cdsapirc
#CDS_API_KEY = None

# Paste your key here only in your private local copy, for example:
CDS_API_KEY = ''

CDS_API_URL = "https://cds.climate.copernicus.eu/api"


# =========================
# DATASET SELECTION
# =========================

# Choose one of: "era5" or "era5land"
PRODUCT = "era5"

# CDS variable name
VARIABLE = "surface_solar_radiation_downwards"


# =========================
# COUNTRY / AREA SELECTION
# =========================

# Country name must match one key in the AREAS dictionary below.
COUNTRY = "argentina"

# Bounding boxes in CDS format: [North, West, South, East]
AREAS = {
    "bolivia": [-9.6, -69.8, -23.0, -57.4],
    "argentina": [-21.7, -73.6, -55.1, -53.5],
    "ecuador": [1.9, -92.0, -5.3, -75.1],
    "peru": [0.1, -81.5, -18.5, -68.5],
    "colombia": [15.9, -81.7, -5.1, -65.9],
}

AREA = AREAS[COUNTRY]


# =========================
# TIME PERIOD
# =========================

# Format: datetime(YYYY, M, D)
START_DATE = datetime(1991, 1, 1)
END_DATE = datetime(1991, 1, 31)


# =========================
# OUTPUT FOLDER
# =========================

# Local, portable output folder. Change BASE_OUTPUT_DIR if needed.
BASE_OUTPUT_DIR = Path("../data") / "cds_downloads"
OUTPUT_DIR = BASE_OUTPUT_DIR / PRODUCT / VARIABLE / COUNTRY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Selected product: {PRODUCT}")
print(f"Selected variable: {VARIABLE}")
print(f"Selected country: {COUNTRY}")
print(f"Selected area: {AREA}")
print(f"Selected period: {START_DATE.date()} to {END_DATE.date()}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")

Selected product: era5
Selected variable: 2m_temperature
Selected country: argentina
Selected area: [-21.7, -73.6, -55.1, -53.5]
Selected period: 1991-01-01 to 1991-01-31
Output folder: /home/python/jupyters/WMO_ATLAS/Notebooks_for_Deliverable_3/data/cds_downloads/era5/2m_temperature/argentina


## Step 3. Functions

Run this cell without editing it.

The functions below:

- map the selected product to the correct CDS dataset name;
- validate the main user inputs before starting the download;
- create the CDS API client;
- download the selected variable month by month.

In [9]:
def get_dataset_name(product):
    """
    Return the CDS dataset name based on the selected product.

    Parameters
    ----------
    product : str
        Product name. Allowed values are "era5" and "era5land".

    Returns
    -------
    str
        CDS dataset name.
    """
    product = product.lower()

    dataset_map = {
        "era5": "reanalysis-era5-single-levels",
        "era5land": "reanalysis-era5-land",
    }

    if product not in dataset_map:
        raise ValueError("PRODUCT must be either 'era5' or 'era5land'.")

    return dataset_map[product]


def validate_inputs(product, variable, country, area, start_date, end_date, output_dir):
    """
    Validate the main notebook inputs before starting the download.
    """
    if product.lower() not in ["era5", "era5land"]:
        raise ValueError("PRODUCT must be either 'era5' or 'era5land'.")

    if not isinstance(variable, str) or not variable.strip():
        raise ValueError("VARIABLE must be a non-empty string.")

    if not isinstance(country, str) or not country.strip():
        raise ValueError("COUNTRY must be a non-empty string.")

    if not isinstance(area, list) or len(area) != 4:
        raise ValueError("AREA must be a list with four values: [North, West, South, East].")

    if start_date > end_date:
        raise ValueError("START_DATE must be earlier than or equal to END_DATE.")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)


def create_cds_client(cds_api_key=None, cds_api_url="https://cds.climate.copernicus.eu/api"):
    """
    Create a CDS API client.

    If cds_api_key is None, cdsapi will use the standard local CDS configuration
    file, usually located at ~/.cdsapirc.
    """
    if cds_api_key is None:
        return cdsapi.Client(url=cds_api_url, progress=False)

    return cdsapi.Client(url=cds_api_url, key=str(cds_api_key), progress=False)


def download_reanalysis_by_month(
    variable,
    start_date,
    end_date,
    output_dir,
    product="era5land",
    area=None,
    cds_client=None,
    overwrite=False,
):
    """
    Download hourly reanalysis data one month at a time from ERA5 or ERA5-Land.

    Each month is saved as a separate NetCDF file.

    Parameters
    ----------
    variable : str
        CDS variable name to download.
    start_date : datetime
        Download start date.
    end_date : datetime
        Download end date.
    output_dir : str or pathlib.Path
        Output directory where files will be saved.
    product : str
        Reanalysis product to use: "era5" or "era5land".
    area : list or None
        Geographic bounding box in CDS format: [North, West, South, East].
    cds_client : cdsapi.Client or None
        Existing CDS client. If None, a new client is created.
    overwrite : bool
        If False, existing files are skipped. If True, existing files are downloaded again.
    """
    dataset_name = get_dataset_name(product)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if cds_client is None:
        cds_client = create_cds_client()

    current_date = start_date.replace(day=1)

    while current_date <= end_date:
        year = current_date.year
        month = current_date.strftime("%m")

        output_file = output_dir / f"{product}_{variable}_{year}_{month}.nc"

        if output_file.exists() and not overwrite:
            print(f"Skipping existing file: {output_file}")
            current_date += timedelta(days=31)
            current_date = current_date.replace(day=1)
            continue

        request_params = {
            "product_type": "reanalysis",
            "data_format": "netcdf",
            "download_format": "unarchived",
            "variable": [variable],
            "year": [str(year)],
            "month": [month],
            "day": [f"{day:02d}" for day in range(1, 32)],
            "time": [f"{hour:02d}:00" for hour in range(24)],
        }

        if area is not None:
            request_params["area"] = area

        print(f"Downloading {product} | {variable} | {year}-{month} ...")
        cds_client.retrieve(dataset_name, request_params, str(output_file))
        print(f"Saved to: {output_file}")

        current_date += timedelta(days=31)
        current_date = current_date.replace(day=1)

## Step 4. Run the download

Run this cell after checking the input parameters above.

Set `OVERWRITE_EXISTING_FILES = True` only if you want to download files again even when they already exist in the output folder.

In [10]:
OVERWRITE_EXISTING_FILES = False

validate_inputs(
    product=PRODUCT,
    variable=VARIABLE,
    country=COUNTRY,
    area=AREA,
    start_date=START_DATE,
    end_date=END_DATE,
    output_dir=OUTPUT_DIR,
)

cds_client = create_cds_client(
    cds_api_key=CDS_API_KEY,
    cds_api_url=CDS_API_URL,
)

download_reanalysis_by_month(
    variable=VARIABLE,
    start_date=START_DATE,
    end_date=END_DATE,
    output_dir=OUTPUT_DIR,
    product=PRODUCT,
    area=AREA,
    cds_client=cds_client,
    overwrite=OVERWRITE_EXISTING_FILES,
)

2026-06-08 09:38:40,615 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-06-08 09:38:40,617 INFO Request ID is bdc7534e-fb22-43cb-a54f-0898e6c544a8
2026-06-08 09:38:40,987 INFO status has been updated to accepted
2026-06-08 09:38:47,679 INFO status has been updated to running
2026-06-08 09:39:58,077 INFO status has been updated to successful


Saved to: ../data/cds_downloads/era5/2m_temperature/argentina/era5_2m_temperature_1991_01.nc


## Notes for adapting this notebook to a new country

To use this notebook for a new country or region:

1. Add a new bounding box to the `AREAS` dictionary.
2. Set `COUNTRY` to the new dictionary key.
3. Check the start and end dates.
4. Run all cells from top to bottom.

The bounding box must follow the CDS order: `[North, West, South, East]`.